In [1]:
# !pip -q install requests beautifulsoup4 pandas spacy transformers tqdm datasets
# !python -m spacy download en_core_web_sm -q


In [2]:
import torch
import os, re, time, datetime as dt, collections, requests, json, textwrap
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup
import spacy
from transformers import pipeline
from tqdm.notebook import tqdm  # Colab‑friendly progress bar

# -------- parameters --------
TICKERS  = ["AMZN","AXP","AMGN","AAPL","BA","CAT","CSCO","CVX","GS","HD",
            "HON","IBM","JNJ","MCD","MMM","MRK","MSFT","NKE","PG","SHW",
            "TRV","UNH","CRM","NVDA","VZ","V","WMT","DIS"]
START    = "2024-04-01"        # inclusive lower bound
END      = "2025-04-16"        # inclusive upper bound
MAX_NEWS = 100                 # google max per query
OUT_DIR  = Path("/content")
UA_HDR   = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/122.0.0.0 Safari/537.36")}
# --------------------------------------------


In [3]:
# 3) load models
import torch


device = 0 if torch.cuda.is_available() else -1
print(f"[INFO] running sentiment pipeline on device {device}")

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)




[INFO] running sentiment pipeline on device 0


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
def google_html(query:str, n:int=100)->list[dict]:
    url = ("https://www.google.com/search"
           f"?q={requests.utils.quote(query)}&gl=us&tbm=nws&num={n}")
    html = requests.get(url, headers=UA_HDR, timeout=10).text
    soup = BeautifulSoup(html, "html.parser")
    hits=[]
    for el in soup.select("div.SoaBEf"):
        hits.append(dict(
            link    = el.find("a")["href"],
            title   = el.select_one("div.MBeuO").get_text(strip=True),
            snippet = el.select_one(".GI74Re").get_text(" ", strip=True),
            date_lab= el.select_one(".LfVVr").get_text(strip=True),
            source  = el.select_one(".NUnG9d span").get_text(strip=True)
        ))
    return hits

def label_to_date(label:str)->dt.date|None:
    label=label.lower(); today=dt.date.today()
    if "hour" in label or "min" in label: return today
    if "day"  in label:
        n=int(re.search(r"\d+", label)[0]); return today-dt.timedelta(days=n)
    try: return dt.datetime.strptime(label[:12], "%b %d, %Y").date()
    except: return None

def scrape_article(url:str)->str:
    try:
        html=requests.get(url, headers=UA_HDR, timeout=10).text
        soup=BeautifulSoup(html,"html.parser")
        return " ".join(p.get_text(' ',strip=True) for p in soup.find_all("p"))
    except: return ""

def clean(txt:str)->str:
    doc=nlp(txt)
    return " ".join(t.lemma_ for t in doc if not t.is_stop and not t.is_punct)

def starscore(txt:str)->float:
    tot=n=0
    for ck in (txt[i:i+500] for i in range(0,len(txt),500)):
        best=max(unwrap(star(ck)), key=lambda d:d["score"])
        tot+=STAR_MAP[best["label"]]; n+=1
    return tot/n if n else 3.0

def bestemo(txt:str)->str:
    cnt=collections.Counter()
    for ck in (txt[i:i+500] for i in range(0,len(txt),500)):
        best=max(unwrap(emo(ck)), key=lambda d:d["score"])
        cnt[best["label"]]+=1
    return cnt.most_common(1)[0][0] if cnt else "neutral"


In [ ]:
# 5) Main loop — per‑ticker & per‑date via RSS → BERT sentiment → daily mean

import xml.etree.ElementTree as ET
import datetime as dt
from tqdm.notebook import tqdm
import pandas as pd
import time

# helper to fetch exactly one day's RSS headlines
def fetch_rss_for_date(ticker: str, date: dt.date):
    after  = date.isoformat()
    before = (date + dt.timedelta(days=1)).isoformat()
    q      = f"{ticker}+after:{after}+before:{before}"
    url    = (
        "https://news.google.com/rss/search"
        f"?q={requests.utils.quote(q)}"
        "&hl=en-US&gl=US&ceid=US:en"
    )
    r = requests.get(url, headers=UA_HDR, timeout=10)
    if r.status_code != 200:
        return []
    root = ET.fromstring(r.content)
    return [
        {"date": after, "text": it.findtext("title","")}
        for it in root.findall(".//item")
    ]

# helper to flatten pipeline output
def unwrap(x):
    while isinstance(x, list) and x:
        x = x[0]
    return x

date_list = pd.date_range(start=START, end=END).date
combined = []

for tic in tqdm(TICKERS, desc="Tickers"):
    # collect date/text records
    records = []
    for pub in date_list:
        hits = fetch_rss_for_date(tic, pub)
        if hits:
            records.extend(hits)
        else:
            # ensure every date appears
            records.append({"date": pub.isoformat(), "text": ""})
        time.sleep(0.2)  # polite

    df = pd.DataFrame(records)
    texts = df["text"].tolist()

    # batch‐classify all texts at once
    preds = sentiment(
        texts,
        batch_size=16,
        truncation=True,
        padding=True,
        max_length=512
    )
    # flatten nested lists (if any)
    flat = [unwrap(p) for p in preds]

    # map to continuous [0,1]:
    #   POSITIVE → score, NEGATIVE → (1 - score)
    df["sentiment_score"] = [
        p["score"] if p["label"] == "POSITIVE" else 1 - p["score"]
        for p in flat
    ]

    # aggregate per‐day
    daily = (
        df.groupby("date", as_index=False)
          .agg(sentiment_score=("sentiment_score", "mean"))
    )

    # save per‐ticker CSV
    out_path = OUT_DIR / f"{tic}_daily_sentiment.csv"
    daily.to_csv(out_path, index=False)
    print(f"[INFO] {tic}: saved {len(daily)} days → {out_path.name}")

    daily["ticker"] = tic
    combined.append(daily)

# optional combined CSV
if combined:
    all_df   = pd.concat(combined, ignore_index=True)
    all_path = OUT_DIR / "all_tickers_daily_sentiment.csv"
    all_df.to_csv(all_path, index=False)
    print(f"[INFO] combined CSV saved → {all_path.name}")
else:
    print("[WARN] no data generated")


Tickers:   0%|          | 0/28 [00:00<?, ?it/s]

[INFO] AMZN: saved 381 days → AMZN_daily_sentiment.csv
